# Conversation Evaluation EDA

End-to-end analysis of the post-training conversation eval. Loads conversations + per-patient
scores from `data/`, runs the stat battery, plots the headline figures.

**Flow:** Setup → Load → Pick a selection (all / best per oracle) → Analyses.
Each analysis cell is independent; rerun any subset after picking a selection.

**Eval scoring** is a separate step — run [Run_Eval.ipynb](Run_Eval.ipynb) first if any model is missing eval CSVs.


## 1. Setup

One-shot import + plot style. Output cells below this point are independent and re-runnable.


In [ ]:
from lib import (
    WORKSPACE_ROOT, ORACLE_METRIC_MAP, EDAConfig, PlotContext, set_plot_style,
    EXPERIMENTS, get_model_names, resolve_paths,
    load_data, combine_data, add_model_metadata_columns, add_patient_characteristics,
    compute_model_order, apply_model_order, build_experiment_palette,
    load_all_eval_results, merge_q1_q2_results, build_test_cases,
    select_best_models_by_own_oracle, filter_to_models, build_merged_metrics,
    format_pvalue, iter_metric_cases, run_full_stats_battery,
    compare_all_vs_baseline, compare_all_pairwise, compare_lookahead,
    plot_metric_by_model, plot_subscales, plot_all_metrics_grid,
    plot_metrics_by_patient_characteristic, plot_correlation_heatmaps,
    plot_session_ending,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

set_plot_style()


## 2. Load Conversations + Eval Scores

Conversations and eval CSVs are loaded together so the rest of the notebook can mix-and-match.
After this cell: `combined_data`, `eval_results`, `test_cases`, `MODEL_ORDER`, `ctx`.


In [ ]:
config = EDAConfig()

# ── Conversations ──
combined_data = combine_data(load_data(resolve_paths()), get_model_names())
combined_data["session_ended_by"] = combined_data["session_ended_by"].apply(
    lambda x: x if x in ["therapist", "patient"] else "None"
)
combined_data["id"] = combined_data["id"].astype(int)
combined_data = add_model_metadata_columns(combined_data)

MODEL_ORDER = compute_model_order(combined_data["Model"].astype(str).tolist())
combined_data = apply_model_order(combined_data, model_order=MODEL_ORDER)
combined_data = add_patient_characteristics(combined_data, id_col="id")

experiment_palette = build_experiment_palette(combined_data)
model_palette = {
    row["Model"]: experiment_palette.get(row["ExperimentGroup"], "gray")
    for _, row in combined_data[["Model", "ExperimentGroup"]].drop_duplicates().iterrows()
}
ctx = PlotContext(
    model_palette=model_palette,
    experiment_palette=experiment_palette,
    hue_col="ExperimentGroup",
    model_order=MODEL_ORDER,
)

# ── Eval scores ──
model_list = combined_data["Model"].astype(str).unique().tolist()
eval_results = load_all_eval_results(config.eval_folders, model_list, MODEL_ORDER)
eval_results["Q1+Q2"] = merge_q1_q2_results(eval_results.get("Q1"), eval_results.get("Q2"))
if eval_results["Q1+Q2"] is not None:
    eval_results["Q1+Q2"] = apply_model_order(
        add_model_metadata_columns(eval_results["Q1+Q2"]), MODEL_ORDER
    )
test_cases = build_test_cases(eval_results, ORACLE_METRIC_MAP)


## 3. TL;DR

Quick orientation: what's loaded, headline mean per metric per group, and which iteration
wins each group's own oracle.


In [ ]:
# What's loaded ─────────────────────────────────────────────────────
group_summary = combined_data.groupby("ExperimentGroup", observed=True).agg(
    conversations=("id", "size"),
    models=("Model", "nunique"),
)
display(
    group_summary.style.set_caption(
        f"Loaded {len(combined_data):,} conversations · "
        f"{combined_data['Model'].nunique()} models · "
        f"{combined_data['ExperimentGroup'].nunique()} groups · "
        f"{len(test_cases)} eval metrics"
    )
)

# Mean per metric per experiment group ────────────────────────────
rows = []
for name, df, col in test_cases:
    rows.append(df.groupby("ExperimentGroup", observed=True)[col].mean().rename(name))
headline = pd.concat(rows, axis=1)
display(headline.style.format("{:.3f}").background_gradient(cmap="Greens").set_caption(
    "Mean per metric × ExperimentGroup"
))


## 4. Pick a Selection

Run **one** of `4a` (all models, every iteration) or `4b` (best iteration per group, judged by
its own training oracle). Both produce: `selected_models`, `selected_test_cases`,
`selected_combined_data`, `selected_model_order`, `selection_label`.


### 4a. All Models — every iteration of every group

In [ ]:
selected_models        = combined_data["Model"].astype(str).unique().tolist()
selected_test_cases    = test_cases
selected_combined_data = combined_data
selected_model_order   = MODEL_ORDER
selection_label        = f"All Models (n={len(selected_models)})"
print(selection_label)


### 4b. Best per Group by Own Oracle

Each DPO group (e.g. `L0_WAI`) is judged by the questionnaire it was trained on.
GRPO_Exp3 is judged by Q1+Q2.


In [ ]:
selected_models, own_oracle_summary = select_best_models_by_own_oracle(
    test_cases, ORACLE_METRIC_MAP,
)
selected_test_cases    = filter_to_models(test_cases, selected_models)
selected_combined_data = filter_to_models(combined_data, selected_models)
selected_model_order   = [m for m in MODEL_ORDER if m in set(selected_models)]
selection_label        = f"Best by Own Oracle (n={len(selected_models)})"

display(
    own_oracle_summary.style
    .format({"Mean": "{:.3f}", "Std": "{:.3f}"})
    .set_caption(selection_label)
)


## 5. Analyses

Each cell is independent — run any subset after picking a selection above.

**Reading the figures and tables:**
- **Bar heights** = mean score across 96 patient permutations.
- **Error bars** = 95% CI.
- **Dotted black line** = Base model mean.
- **Significance stars:** \* p<0.05, \*\* p<0.01, \*\*\* p<0.001.
- **Cohen's d:** <0.2 negligible, <0.5 small, <0.8 medium, ≥0.8 large.
- **η²:** >0.01 small, >0.06 medium, >0.14 large.
- **q_fdr:** BH-FDR adjusted p-value (significant if <0.05).


### 5.1 Metric Overview Grid

In [ ]:
plot_all_metrics_grid(
    selected_test_cases, group_by="Model",
    model_order=selected_model_order, ylim="auto", padding=0.25, ctx=ctx,
    figsize=(16, 16),
)
plt.suptitle(f"Metric Overview — {selection_label}", fontsize=14, fontweight="bold", y=1.01)
plt.show()


### 5.2 WAI-SR Subscales

In [ ]:
wai_df = eval_results.get("WAI-SR")
wai_models = {m for n, df, _ in selected_test_cases if n == "WAI-SR" and df is not None
              for m in df["Model"].astype(str).unique()}
if wai_df is not None and wai_models:
    plot_df = apply_model_order(wai_df[wai_df["Model"].isin(wai_models)].copy(), MODEL_ORDER)
    plot_subscales(
        plot_df,
        subscale_cols=["WAI_Goal_Mean", "WAI_Task_Mean", "WAI_Bond_Mean"],
        subscale_titles=["Goal", "Task", "Bond"],
        ylim="auto", suptitle=f"WAI-SR Subscales — {selection_label}",
        model_order=[m for m in selected_model_order if m in plot_df["Model"].unique()],
        ctx=ctx,
    )
else:
    print("WAI-SR data not available.")


### 5.3 MITI Subscales

In [ ]:
miti_df = eval_results.get("MITI")
miti_models = {m for n, df, _ in selected_test_cases if n == "MITI" and df is not None
               for m in df["Model"].astype(str).unique()}
if miti_df is not None and miti_models:
    plot_df = apply_model_order(miti_df[miti_df["Model"].isin(miti_models)].copy(), MODEL_ORDER)
    plot_subscales(
        plot_df,
        subscale_cols=["MITI1_CultivatingChangeTalk", "MITI2_SofteningSustainTalk",
                       "MITI3_Partnership", "MITI4_Empathy"],
        subscale_titles=["Change Talk", "Softening Sustain", "Partnership", "Empathy"],
        figsize=(22, 5), ylim="auto",
        suptitle=f"MITI Subscales — {selection_label}",
        model_order=[m for m in selected_model_order if m in plot_df["Model"].unique()],
        ctx=ctx,
    )
else:
    print("MITI data not available.")


### 5.4 Statistical Battery

Three layers: omnibus (any difference?) → vs Base (which models beat Base?) → pairwise.


#### 5.4a Omnibus — does any model differ within each metric?

In [ ]:
stats_results = run_full_stats_battery(selected_test_cases, groupby_levels=["Model"], baseline="Base")

rows = []
for key, payload in stats_results.items():
    metric = key.rsplit("_", 1)[0]
    a = payload.get("anova") or {}
    k = payload.get("kruskal") or {}
    p_a = a.get("p_value", 1.0)
    rows.append({
        "Metric": metric,
        "F (ANOVA)":         a.get("F_statistic", float("nan")),
        "p-val (ANOVA)":     format_pvalue(p_a),
        "η²":                a.get("eta_squared", float("nan")),
        "H (Kruskal-Wallis)": k.get("H_statistic", float("nan")),
        "p-val (K-W)":       format_pvalue(k.get("p_value", 1.0)),
        "Sig?":              "✓" if p_a < 0.05 else "",
    })
if rows:
    display(
        pd.DataFrame(rows).set_index("Metric").style
        .format({"F (ANOVA)": "{:.2f}", "η²": "{:.3f}", "H (Kruskal-Wallis)": "{:.2f}"})
        .background_gradient(subset=["η²"], cmap="RdYlGn_r")
        .set_caption(f"5.4a Omnibus — {selection_label}")
    )


#### 5.4b Each Model vs Base — Mann-Whitney U + BH-FDR

In [ ]:
vs_base = compare_all_vs_baseline(selected_test_cases, baseline="Base")
if vs_base.empty:
    print("No data for vs-baseline comparison.")
else:
    n_sig = int(vs_base["Significant"].sum())
    print(f"vs-Base: {n_sig}/{len(vs_base)} tests significant after BH-FDR (q < 0.05)")
    for metric in vs_base["Metric"].unique():
        sig = vs_base[(vs_base["Metric"] == metric) & vs_base["Significant"]].copy()
        if sig.empty:
            continue
        display(
            sig.drop(columns=["Metric", "Significant"]).reset_index(drop=True).style
            .format({
                "Mean": "{:.3f}", "Baseline_Mean": "{:.3f}", "Delta_Mean": "{:.3f}",
                "U": "{:.0f}", "p_raw": "{:.4f}", "q_fdr": "{:.4f}", "Cohens_d": "{:.3f}",
            })
            .set_caption(f"5.4b {metric} — {len(sig)} significant vs Base ({selection_label})")
        )


#### 5.4c All Pairwise — Mann-Whitney U + BH-FDR per metric

In [ ]:
pairwise = compare_all_pairwise(selected_test_cases)
if pairwise.empty:
    print("No significant pairwise differences after BH-FDR.")
else:
    print(f"Pairwise: {len(pairwise)} significant pairs after BH-FDR (q < 0.05)")
    for metric in pairwise["Metric"].unique():
        sl = pairwise[pairwise["Metric"] == metric].copy()
        if sl.empty:
            continue
        display(
            sl.drop(columns=["Metric"]).reset_index(drop=True).style
            .format({
                "Mean_A": "{:.3f}", "Mean_B": "{:.3f}", "Delta": "{:.3f}",
                "U": "{:.0f}", "p_raw": "{:.4f}", "q_fdr": "{:.4f}", "Cohens_d": "{:.3f}",
            })
            .set_caption(f"5.4c {metric} — {len(sl)} significant pairs ({selection_label})")
        )


### 5.5 Model Rankings

Per-metric mean / std / rank, plus an `Avg Rank` column averaged across all metrics. Lower
average rank = better overall. Base is pinned to the top for reference.


In [ ]:
ranking = None
for name, df, col in iter_metric_cases(selected_test_cases):
    g = df.groupby("Model", observed=True)[col].agg(["mean", "std"]).reset_index()
    g["rank"] = g["mean"].rank(ascending=False, method="min").astype(int)
    g = g.rename(columns={"mean": f"{name} Mean", "std": f"{name} Std", "rank": f"{name} Rank"})
    ranking = g if ranking is None else ranking.merge(g, on="Model", how="outer")

if ranking is not None:
    rank_cols = [c for c in ranking.columns if c.endswith(" Rank")]
    mean_cols = [c for c in ranking.columns if c.endswith(" Mean")]
    std_cols  = [c for c in ranking.columns if c.endswith(" Std")]
    ranking["Avg Rank"] = ranking[rank_cols].mean(axis=1).round(1)
    ranking = ranking[["Model", "Avg Rank"] + mean_cols + rank_cols + std_cols]
    ranking = ranking.sort_values("Avg Rank").reset_index(drop=True)
    base_mask = ranking["Model"].astype(str).str.lower().eq("base")
    ranking = pd.concat([ranking[base_mask], ranking[~base_mask]]).reset_index(drop=True)
    display(
        ranking.style
        .format({c: "{:.3f}" for c in mean_cols + std_cols} | {"Avg Rank": "{:.1f}"})
        .background_gradient(subset=["Avg Rank"], cmap="RdYlGn_r", vmin=1, vmax=len(ranking))
        .set_caption(f"Model rankings — {selection_label}")
    )


### 5.6 Lookahead Comparison (L0 vs L5)

Does the K-turn look-ahead during oracle scoring produce different models?


In [ ]:
la_results = compare_lookahead(selected_test_cases)
rows = []
for metric, breakdown in la_results.items():
    r = breakdown.get("overall", {})
    rows.append({
        "Metric": metric,
        "L0 mean": round(r.get("group1_mean", float("nan")), 3),
        "L5 mean": round(r.get("group2_mean", float("nan")), 3),
        "U":      round(r.get("U_statistic", float("nan")), 1),
        "p":      format_pvalue(r.get("p_value", 1.0)),
        "d":      round(r.get("cohens_d", float("nan")), 3),
        "effect": r.get("effect_size", ""),
        "sig":    "*" if r.get("significant") else "",
    })
if rows:
    display(pd.DataFrame(rows).set_index("Metric").style.set_caption(
        f"L0 vs L5 — Mann-Whitney U ({selection_label})"
    ))
else:
    print("No L0/L5 data available for comparison.")


### 5.7 Cross-Metric Correlation

In [ ]:
merged = build_merged_metrics(selected_test_cases)
if merged is not None:
    plot_correlation_heatmaps(merged, title=f"Cross-Metric Correlation — {selection_label}")
else:
    print("Need at least 2 metrics for correlation analysis.")


### 5.8 Metrics by Patient Cooperation Level

In [ ]:
metric_dfs  = {n: df for n, df, _ in iter_metric_cases(selected_test_cases)}
metric_cols = {n: c  for n, _, c  in iter_metric_cases(selected_test_cases)}
plot_metrics_by_patient_characteristic(
    metric_dfs, metric_cols, "cooperation_level",
    palette=ctx.experiment_palette, model_palette=ctx.model_palette,
    figsize=(35, 15), ylim="auto", ctx=ctx,
)


### 5.9 Conversation Length

In [ ]:
present = set(selected_combined_data["Model"].astype(str).unique())
order = [m for m in selected_model_order if m in present]
plot_metric_by_model(
    selected_combined_data, metric="conversation_length",
    title=f"Avg Conversation Length — {selection_label}",
    ylim=(15, 40), model_order=order, ctx=ctx,
)


### 5.10 Session Ending Patterns

In [ ]:
present = set(selected_combined_data["Model"].astype(str).unique())
order = [m for m in selected_model_order if m in present]
plot_session_ending(
    selected_combined_data, palette=ctx.experiment_palette,
    figsize=(24, 8), model_order=order,
)
